# H&M Recommender — Exploratory Data Analysis
**Data Engineer:** Akash  
**Dataset:** H&M Personalized Fashion Recommendations (Kaggle)

## Contents
1. Data loading & basic stats
2. Transaction volume over time
3. Purchase frequency distribution
4. Top product categories & garment groups
5. Customer age & demographics
6. Price distribution
7. Colour distribution
8. Sparsity analysis of the user-item matrix

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW = Path('../data/raw')
PROC = Path('../data/processed')

## 1. Data Loading

In [ ]:
txn = pd.read_csv(RAW / 'transactions_train.csv', dtype={'customer_id': str, 'article_id': str})
txn['t_dat'] = pd.to_datetime(txn['t_dat'])

art = pd.read_csv(RAW / 'articles.csv', dtype={'article_id': str})
cust = pd.read_csv(RAW / 'customers.csv', dtype={'customer_id': str})

print(f'Transactions : {len(txn):>12,}')
print(f'Articles     : {len(art):>12,}')
print(f'Customers    : {len(cust):>12,}')
print(f'Date range   : {txn.t_dat.min().date()} → {txn.t_dat.max().date()}')

## 2. Transaction Volume Over Time

In [ ]:
monthly = txn.resample('ME', on='t_dat').size().reset_index(name='purchases')

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(monthly['t_dat'], monthly['purchases'], alpha=0.3, color='#e63946')
ax.plot(monthly['t_dat'], monthly['purchases'], color='#e63946', lw=2)
ax.set_title('Monthly Transaction Volume', fontsize=14)
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

## 3. Purchase Frequency Distribution

In [ ]:
purchase_counts = txn.groupby('customer_id').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(purchase_counts, bins=60, color='#457b9d', edgecolor='white', log=True)
axes[0].set_title('Purchases per Customer (log scale)')
axes[0].set_xlabel('Number of purchases')
axes[0].set_ylabel('Count (log)')

cdf = purchase_counts.value_counts().sort_index().cumsum() / len(purchase_counts)
axes[1].plot(cdf.index[:200], cdf.values[:200], color='#e63946', lw=2)
axes[1].axvline(5, color='grey', ls='--', label='Min threshold (5)')
axes[1].set_title('CDF — Purchases per Customer')
axes[1].set_xlabel('Number of purchases')
axes[1].set_ylabel('Cumulative fraction')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Customers with ≥5 purchases: {(purchase_counts>=5).sum():,} ({(purchase_counts>=5).mean()*100:.1f}%)')

## 4. Top Product Categories

In [ ]:
merged = txn.merge(art[['article_id','product_type_name','index_group_name','garment_group_name']], on='article_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_types = merged['product_type_name'].value_counts().head(15)
top_types.sort_values().plot(kind='barh', ax=axes[0], color='#457b9d')
axes[0].set_title('Top 15 Product Types')

top_groups = merged['index_group_name'].value_counts().head(10)
top_groups.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
                colors=sns.color_palette('pastel'))
axes[1].set_title('Index Group Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 5. Customer Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cust['age'].dropna().plot(kind='hist', bins=40, ax=axes[0], color='#e63946', edgecolor='white')
axes[0].set_title('Age Distribution')

cust['club_member_status'].value_counts().plot(kind='bar', ax=axes[1], color='#457b9d', edgecolor='white', rot=0)
axes[1].set_title('Club Member Status')

cust['fashion_news_frequency'].value_counts().plot(kind='bar', ax=axes[2], color='#a8dadc', edgecolor='white', rot=0)
axes[2].set_title('Fashion News Frequency')

plt.tight_layout()
plt.show()

## 6. Price Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
txn['price'].dropna().clip(upper=txn['price'].quantile(0.99)).plot(
    kind='hist', bins=60, ax=ax, color='#1d3557', edgecolor='white')
ax.set_title('Transaction Price Distribution (clipped at 99th percentile)')
ax.set_xlabel('Price (€)')
plt.tight_layout()
plt.show()

print(txn['price'].describe())

## 7. Colour Distribution

In [ ]:
colour_counts = merged['colour_group_name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
colour_counts.sort_values().plot(kind='barh', ax=ax, color='#a8dadc', edgecolor='white')
ax.set_title('Top 15 Colour Groups in Transactions')
plt.tight_layout()
plt.show()

## 8. Sparsity Analysis (Filtered 6-month window)

In [ ]:
cutoff = txn['t_dat'].max() - pd.DateOffset(months=6)
txn_6m = txn[txn['t_dat'] >= cutoff]
counts = txn_6m.groupby('customer_id').size()
txn_active = txn_6m[txn_6m['customer_id'].isin(counts[counts >= 5].index)]

n_users = txn_active['customer_id'].nunique()
n_items = txn_active['article_id'].nunique()
n_interactions = txn_active.groupby(['customer_id','article_id']).ngroups
sparsity = 1 - n_interactions / (n_users * n_items)

print(f'Filtered dataset (6m, ≥5 purchases):')
print(f'  Users        : {n_users:,}')
print(f'  Articles     : {n_items:,}')
print(f'  Interactions : {n_interactions:,}')
print(f'  Sparsity     : {sparsity*100:.4f}%')